# Задание 4. Тренд, сезонность и автокорреляция

В работе исследуются два временных ряда из `TimeSeries.xls`:

- давление `P` — анализ тренда и стационарности;
- расход `Q` — анализ суточной сезонности.

Основные инструменты: графики временного ряда, ACF, удаление тренда, разности, ADF-тест и несколько способов оценки сезонной составляющей.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from IPython.display import display
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller

np.set_printoptions(suppress=True)

## Загрузка данных

In [ ]:
sheets = pd.read_excel("TimeSeries.xls", sheet_name=None)
sheet_names = list(sheets)

# Сохраняем исходную схему чтения данных из лабораторной работы.
data_p = sheets[sheet_names[0]].to_numpy().ravel()[1:].astype(float)
data_q = sheets[sheet_names[1]].to_numpy().ravel()[1::2][1:].astype(float)

# Давление измеряется через 4 часа, расход — через 2 часа.
time_p = np.arange(len(data_p)) * 4
time_q = np.arange(len(data_q)) * 2

print(f"Наблюдений давления: {len(data_p)}")
print(f"Наблюдений расхода:   {len(data_q)}")

---

# 1. Давление: тренд и стационарность

Сначала посмотрим на исходный ряд и его автокорреляционную функцию. Если средний уровень ряда систематически меняется со временем, а ACF затухает медленно, это признак нестационарности.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time_p, data_p)
ax.set_title("Давление во времени")
ax.set_xlabel("Время, ч")
ax.set_ylabel("P")
ax.grid()
plt.show()

plot_acf(data_p, lags=200)
plt.title("ACF исходного ряда давления")
plt.grid()
plt.show()

### Вывод по исходному ряду

На графике исходного давления виден устойчивый нисходящий тренд. ACF остаётся высокой на большом числе лагов и затухает очень медленно. Поэтому исходный ряд нельзя считать стационарным только по визуальной диагностике.

## 1.1. Удаление линейного тренда методом наименьших квадратов

В исходной версии ноутбука в `np.polyfit` использовалась степень `2`, хотя в задании требуется **линейный** тренд. Здесь используется полином первой степени.

In [ ]:
coef = np.polyfit(time_p, data_p, deg=1)
trend_p = np.polyval(coef, time_p)
data_p_detrended = data_p - trend_p

print(f"Линейный тренд: P(t) = {coef[0]:.8f} * t + {coef[1]:.6f}")

plt.figure(figsize=(14, 5))
plt.plot(time_p, data_p, label="Исходный ряд")
plt.plot(time_p, trend_p, "--", label="Линейный тренд")
plt.title("Оценка линейного тренда")
plt.xlabel("Время, ч")
plt.ylabel("P")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(time_p, data_p_detrended)
plt.axhline(0, linewidth=1)
plt.title("Остатки после удаления линейного тренда")
plt.xlabel("Время, ч")
plt.grid()
plt.show()

plot_acf(data_p_detrended, lags=200)
plt.title("ACF после удаления линейного тренда")
plt.grid()
plt.show()

Удаление линейного тренда убирает основное долгосрочное снижение среднего уровня. При этом остатки всё ещё могут сохранять краткосрочную автокорреляцию: отсутствие тренда само по себе не означает, что наблюдения стали независимыми.

## 1.2. Первые разности

Второй способ убрать нестационарность по уровню:

\[
\Delta P_t = P_t - P_{t-1} = (1-L)P_t.
\]

In [ ]:
data_p_diff = np.diff(data_p)
time_p_diff = time_p[1:]

plt.figure(figsize=(14, 4))
plt.plot(time_p_diff, data_p_diff)
plt.axhline(0, linewidth=1)
plt.title("Первые разности давления")
plt.xlabel("Время, ч")
plt.grid()
plt.show()

plot_acf(data_p_diff, lags=100)
plt.title("ACF первых разностей давления")
plt.grid()
plt.show()

После перехода к первым разностям длительная положительная автокорреляция исходного ряда исчезает. Это существенно ближе к стационарному поведению, хотя на нескольких малых лагах остаётся краткосрочная зависимость.

## 1.3. Расширенный тест Дики—Фуллера

Результат ADF зависит от того, какие детерминированные компоненты включены в тест:

- `n` — без константы;
- `c` — с константой;
- `ct` — с константой и линейным трендом;
- `ctt` — с константой, линейным и квадратичным трендом.

Поскольку на графике исходного ряда заметен тренд, модели без детерминированной части нельзя интерпретировать отдельно от визуальной структуры ряда.

In [ ]:
def adf_row(series, regression, name):
    stat, p_value, crit_values, store = adfuller(
        series,
        regression=regression,
        autolag="AIC",
        store=True,
        regresults=True,
    )
    return {
        "model": name,
        "regression": regression,
        "ADF": stat,
        "p_value": p_value,
        "used_lag": store.usedlag,
        "n_obs": store.nobs,
        "critical_5%": crit_values["5%"],
    }, store


specifications = [
    ("AR: без константы", "n"),
    ("ARD: константа", "c"),
    ("TS: константа + тренд", "ct"),
    ("Квадратичный тренд", "ctt"),
]

rows = []
stores = {}
for name, regression in specifications:
    row, store = adf_row(data_p, regression, name)
    rows.append(row)
    stores[regression] = store

adf_table = pd.DataFrame(rows)
display(adf_table.round(5))

print("\nВспомогательная регрессия для TS-модели:")
print(stores["ct"].resols.summary())

diff_row, _ = adf_row(data_p_diff, "c", "Первые разности")
print("\nADF для первых разностей:")
display(pd.DataFrame([diff_row]).round(5))

### Вывод по стационарности давления

Для исходного ряда вывод ADF чувствителен к выбранной детерминированной части, что ожидаемо при заметном тренде. Поэтому решение лучше принимать не по одному `p-value`, а вместе с графиком и ACF.

Первые разности убирают долгосрочный тренд значительно надёжнее: после дифференцирования ACF быстро возвращается к нулю. Для дальнейшего моделирования такой ряд является более естественной стационарной формой.

---

# 2. Расход: сезонность

Шаг между наблюдениями расхода равен 2 часам. Поэтому суточный цикл соответствует

\[
24 / 2 = 12
\]

наблюдениям. Проверим, виден ли лаг `k = 12` в ACF.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(time_q, data_q, "-o", markersize=3)
plt.title("Расход во времени")
plt.xlabel("Время, ч")
plt.ylabel("Q")
plt.grid()
plt.show()

plot_acf(data_q, lags=50)
plt.axvline(12, linestyle="--", label="k = 12")
plt.title("ACF исходного ряда расхода")
plt.legend()
plt.grid()
plt.show()

В ACF заметны повторяющиеся пики около лагов `12`, `24`, `36`, `48`. Это соответствует выраженной периодической составляющей с периодом **12 наблюдений**, то есть примерно **24 часа**.

## 2.1. Удаление тренда первыми разностями

In [ ]:
data_q_diff = np.diff(data_q)
time_q_diff = time_q[1:]

plt.figure(figsize=(14, 5))
plt.plot(time_q_diff, data_q_diff, "-o", markersize=3)
plt.axhline(0, linewidth=1)
plt.title("Первые разности расхода")
plt.xlabel("Время, ч")
plt.grid()
plt.show()

plot_acf(data_q_diff, lags=50)
plt.axvline(12, linestyle="--", label="k = 12")
plt.title("ACF первых разностей расхода")
plt.legend()
plt.grid()
plt.show()

Первые разности уменьшают медленное изменение уровня ряда, но сезонный пик на лаге `12` сохраняется. Значит, тренд и сезонность — разные компоненты, и одной обычной разности недостаточно для удаления суточной периодики.

## 2.2. Сезонная компонента через ряд Фурье

Для периода `12` нельзя бездумно добавлять гармоники `1 ... 12`: частоты выше Найквиста дублируют уже существующие столбцы, а синус на половине периода вырождается. В исходной версии это приводило к почти сингулярной матрице и огромному condition number.

Ниже строится полный **невырожденный** Fourier-базис для периода 12: пары `sin/cos` для гармоник 1–5 и отдельный cosine-компонент для шестой гармоники.

In [ ]:
def fourier_design(n, period):
    t = np.arange(n, dtype=float)
    columns = [np.ones(n)]
    names = ["const"]

    # Для period=12 получаем пары sin/cos для гармоник 1..5.
    for harmonic in range(1, (period + 1) // 2):
        angle = 2 * np.pi * harmonic * t / period
        columns.extend([np.cos(angle), np.sin(angle)])
        names.extend([f"cos_{harmonic}", f"sin_{harmonic}"])

    # При чётном периоде частота Найквиста имеет только cosine-компонент.
    if period % 2 == 0:
        harmonic = period // 2
        angle = 2 * np.pi * harmonic * t / period
        columns.append(np.cos(angle))
        names.append(f"cos_{harmonic}")

    return np.column_stack(columns), names


period = 12
X_fourier, fourier_names = fourier_design(len(data_q_diff), period)

print("Размер матрицы:", X_fourier.shape)
print("Ранг матрицы:", np.linalg.matrix_rank(X_fourier))
print("Condition number:", np.linalg.cond(X_fourier))

fourier_model = sm.OLS(data_q_diff, X_fourier).fit()
seasonal_fourier = fourier_model.predict(X_fourier)
residuals_fourier = data_q_diff - seasonal_fourier

fourier_coefficients = pd.DataFrame({
    "term": fourier_names,
    "coef": fourier_model.params,
    "p_value": fourier_model.pvalues,
})
display(fourier_coefficients.round(4))

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(data_q_diff, label="Ряд после обычной разности")
plt.plot(seasonal_fourier, label="Сезонная компонента Фурье")
plt.title("Оценка сезонной компоненты рядом Фурье")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(14, 4))
plt.plot(residuals_fourier)
plt.axhline(0, linewidth=1)
plt.title("Остатки после удаления Fourier-сезонности")
plt.grid()
plt.show()

plot_acf(residuals_fourier, lags=90)
plt.title("ACF остатков после Fourier-модели")
plt.grid()
plt.show()

var_fourier = np.var(residuals_fourier)
print(f"Дисперсия остатков: {var_fourier:.6f}")

Полный Fourier-базис содержит ровно столько степеней свободы, сколько нужно для произвольного сезонного профиля из 12 временных слотов. Поэтому он не должен страдать от мультиколлинеарности, которая возникала в исходной матрице из 25 дублирующихся признаков.

## 2.3. Индикаторы времени суток

Второй способ — создать 12 dummy-переменных для временных слотов `0, 2, ..., 22` часов.

Модель строится **без дополнительной константы**: каждый коэффициент напрямую оценивает среднее значение ряда для соответствующего временного слота. Удалять индикаторы только потому, что их среднее статистически не отличается от нуля, здесь не требуется — задача состоит в оценке полного сезонного профиля.

In [ ]:
slot_idx = ((time_q_diff % 24) // 2).astype(int)

X_indicators = np.eye(period)[slot_idx]
indicator_model = sm.OLS(data_q_diff, X_indicators).fit()

seasonal_indicators = indicator_model.predict(X_indicators)
residuals_indicators = data_q_diff - seasonal_indicators

indicator_table = pd.DataFrame({
    "hour": np.arange(period) * 2,
    "coefficient": indicator_model.params,
    "p_value": indicator_model.pvalues,
})
display(indicator_table.round(4))

plt.figure(figsize=(14, 5))
plt.plot(data_q_diff, label="Ряд после обычной разности")
plt.plot(seasonal_indicators, label="Сезонная компонента по индикаторам")
plt.title("Сезонность по времени суток")
plt.legend()
plt.grid()
plt.show()

plot_acf(residuals_indicators, lags=90)
plt.title("ACF остатков после индикаторной модели")
plt.grid()
plt.show()

var_indicators = np.var(residuals_indicators)
print(f"Дисперсия остатков: {var_indicators:.6f}")

## 2.4. Среднее значение для каждого времени суток

Если в индикаторной модели нет общей константы, её МНК-коэффициенты должны совпадать со средними значениями ряда внутри соответствующих временных слотов. Проверим это напрямую.

In [ ]:
slot_means = np.array([
    data_q_diff[slot_idx == slot].mean()
    for slot in range(period)
])

seasonal_means = slot_means[slot_idx]
residuals_means = data_q_diff - seasonal_means

comparison = pd.DataFrame({
    "hour": np.arange(period) * 2,
    "OLS_indicator": indicator_model.params,
    "slot_mean": slot_means,
    "difference": indicator_model.params - slot_means,
})
display(comparison.round(8))

print(
    "Максимальное отличие OLS-коэффициента от среднего:",
    np.max(np.abs(indicator_model.params - slot_means)),
)

plt.figure(figsize=(14, 5))
plt.plot(data_q_diff, label="Ряд после обычной разности")
plt.plot(seasonal_means, label="Сезонная компонента по средним")
plt.title("Сезонность по средним значениям времени суток")
plt.legend()
plt.grid()
plt.show()

plot_acf(residuals_means, lags=90)
plt.title("ACF остатков после вычитания средних")
plt.grid()
plt.show()

var_means = np.var(residuals_means)
print(f"Дисперсия остатков: {var_means:.6f}")

Индикаторная модель и метод средних — две записи одной и той же оценки сезонного профиля. Поэтому их fitted values и дисперсии остатков должны совпадать с точностью до численного округления.

## 2.5. Сезонные разности

Последний способ использует сезонную разность

\[
\Delta_{12}q_t = q_t - q_{t-12} = (1-L^{12})q_t.
\]

Она удаляет компоненту, повторяющуюся через сутки, но одновременно увеличивает шум, потому что каждая новая точка является разностью двух наблюдений.

In [ ]:
k = 12
seasonal_diff = data_q_diff[k:] - data_q_diff[:-k]

plt.figure(figsize=(14, 4))
plt.plot(seasonal_diff)
plt.axhline(0, linewidth=1)
plt.title(r"Сезонные разности $\Delta_{12}q$")
plt.grid()
plt.show()

plot_acf(seasonal_diff, lags=90)
plt.title("ACF сезонных разностей")
plt.grid()
plt.show()

var_seasonal_diff = np.var(seasonal_diff)
print(f"Дисперсия: {var_seasonal_diff:.6f}")

## 2.6. Сравнение методов

In [ ]:
variance_comparison = pd.DataFrame({
    "method": [
        "Fourier-базис",
        "Индикаторы времени суток",
        "Средние по времени суток",
        "Сезонные разности",
    ],
    "variance": [
        var_fourier,
        var_indicators,
        var_means,
        var_seasonal_diff,
    ],
}).sort_values("variance")

display(variance_comparison)

# Итог

1. **Ряд давления имеет выраженный нисходящий тренд.** Медленно затухающая ACF исходного ряда согласуется с нестационарностью по уровню.
2. **Линейное детрендирование и первые разности решают разные задачи.** После удаления линейного тренда остаётся краткосрочная зависимость, а первые разности значительно сильнее убирают длительную инерцию.
3. **ADF нужно интерпретировать вместе с детерминированной частью модели.** Для трендового ряда спецификации `n`, `c`, `ct`, `ctt` могут приводить к разным формальным выводам.
4. **В расходе есть суточная сезонность.** При шаге 2 часа основной период равен `k = 12`; это видно по повторяющимся пикам ACF.
5. **Обычная первая разность не удаляет сезонность полностью.** После неё лаг 12 остаётся заметным.
6. **Fourier-базис, индикаторы и средние дают явную модель сезонного профиля.** Корректный Fourier-базис для периода 12 должен быть невырожденным; дублирующиеся гармоники использовать нельзя.
7. **Индикаторы без константы и средние по временным слотам эквивалентны.**
8. **Сезонные разности — более грубый способ.** Они не требуют явной оценки сезонного профиля, но могут повышать дисперсию остаточного ряда.

Таким образом, в этой работе тренд и сезонность рассматриваются отдельно: для давления основная проблема — изменение уровня во времени, а для расхода после обычного дифференцирования остаётся выраженная суточная периодичность.